# Generate samplesheet for trisomy detection

## 20260123

Generate a samplesheet for all cfDNA samples before, data batch:

2025: before0819/0930/1015/1031/1106/1122/1128/1219/1226

2026: 0107/0109/0115

In [50]:
# Clear up old samplesheet

import pandas as pd

black_list = [
    'PTAY0691P6S1', 
    'PTAY0680P8S1',
    'PTAY0703P9S1',
    'PTAY0560P8S1',
    'PTAY1179P10S1', 
    'PTAY1219P12S1',
    'D5',
    'D6',
    'F7',
    'M8'
]

meta_df_list = []
mqres_df_list = []

# mqres_to_zscore and meta
for batch in ['20251107', '20251113', '20251122', '20251219', '20251226']:
    mqres_df = pd.read_csv(f'../../snp_nipt/data/220k_panel/mqres_to_pileup/{batch}_mqres_samplesheet.csv').dropna(axis=0)

    # Combine {sample}C and {sample}A together
    if batch == '20260121':
        mqres_df['sample'] = mqres_df['sample'].apply(lambda x: x[:-1] if x.endswith(('C', 'A')) else x)

    mqres_df_list.append(mqres_df)

meta_df = pd.read_csv('../../snp_nipt/results/220k_panel/summary/usable_clean_samplesheet.csv', usecols=['sample', 'label', 'week']).dropna(axis=0)
meta_df_list.append(meta_df)

for batch in ['20260109', '20260121', '20260115']:
    meta_df = pd.read_csv(f'../../snp_nipt/data/220k_panel/mqres_to_pileup/{batch}_meta.tsv', sep='\t', usecols=['sample', 'label', 'week']).dropna(axis=0)
    mqres_df = pd.read_csv(f'../../snp_nipt/data/220k_panel/mqres_to_pileup/{batch}_mqres_samplesheet.csv').dropna(axis=0)

    # Combine {sample}C and {sample}A together
    if batch == '20260121':
        meta_df['sample'] = meta_df['sample'].apply(lambda x: x[:-1] if x.endswith(('C', 'A')) else x)
        mqres_df['sample'] = mqres_df['sample'].apply(lambda x: x[:-1] if x.endswith(('C', 'A')) else x)

    # Combine {sample}_A and {sample}_X together
    if batch == '20260115':
        mqres_df['sample'] = mqres_df['sample'].apply(lambda x: x[:-2] if x.endswith(('A', 'X')) else x)

    meta_df_list.append(meta_df)
    mqres_df_list.append(mqres_df)

meta_df = pd.read_csv('../../snp_nipt/data/220k_panel/mqres_to_pileup/dropped_meta.tsv', sep='\t', usecols=['sample', 'label', 'week']).dropna(axis=0)
meta_df_list.append(meta_df)

meta_df = pd.concat(meta_df_list, axis=0)
mqres_df = pd.concat(mqres_df_list, axis=0)

meta_df = meta_df[~meta_df['sample'].isin(black_list)].reset_index(drop=True)
mqres_df = mqres_df[~mqres_df['sample'].isin(black_list)].reset_index(drop=True)

mqres_df.rename(columns={'bam': 'clean_bam', 'txt': 'deconv_res'}, inplace=True)
mqres_df = mqres_df[['sample', 'clean_bam', 'deconv_res']]

meta_df.to_csv('../../snp_nipt/data/220k_panel/mqres_to_zscore/20260123_meta.tsv', sep='\t', index=False)
mqres_df.to_csv('../../snp_nipt/data/220k_panel/mqres_to_zscore/20260123_mqres_samplesheet.csv', index=False)

early_sample_list = meta_df[meta_df['week'] <= 12]['sample'].unique()
middle_sample_list = meta_df[meta_df['week'] > 12]['sample'].unique()

mqres_df[mqres_df['sample'].isin(early_sample_list)].to_csv('../../snp_nipt/data/220k_panel/mqres_to_zscore/20260123_early_mqres_samplesheet.csv', index=False)
mqres_df[mqres_df['sample'].isin(middle_sample_list)].to_csv('../../snp_nipt/data/220k_panel/mqres_to_zscore/20260123_middle_mqres_samplesheet.csv', index=False)

## 20260127

Generate a samplesheet for resequenced 4 samples, 3 unknown samples and 6 unknown samples in different experiment conditions(_A, _H).

Data batch: 20260120

Resequenced samples in batch: 20250930/20251015/20251226

In [19]:
reseq_sample_list = [
    'PTAY0577P9S1',
    'PTAY0620P8S1',
    'PTAY1024P8S1',
    'PTAY1186P8S1'
]

data_batch = [
    '20250930',
    '20251015',
    '20251226',
    '20260120'
]

# mqres samplesheet
import pandas as pd
import os
import glob
import re

meta = []

for batch in data_batch:
    mqres_dir_path = f'/appsnew/home/myli/lustre1/bert/DNA_5mC_analysis_pipeline/output/{batch}-XML/bwameth_results/mq_downstream'

    pattern = re.compile(r"^(?P<sample>[^_]+)_input.bed.gz_testres$")

    for res_dir in os.listdir(mqres_dir_path):
        match = pattern.match(res_dir)

        if match:
            sample = match.group('sample')
            if sample in reseq_sample_list:          
                txt_path = os.path.join(mqres_dir_path, res_dir, 'test_res.txt')
                bam_path = os.path.join(mqres_dir_path, '../bam_rmdup', f'{sample}.clean.bam')

                sample = sample.split('-')[0]

                if os.path.exists(txt_path) and os.path.exists(bam_path):
                    meta.append({
                        'sample': sample,
                        'deconv_res': txt_path,
                        'clean_bam': bam_path
                    })
                else:
                    print(bam_path)

    pattern = re.compile(r"single_end_(?P<sample>.*)_input.bed.gz_testres$")

    for res_dir in os.listdir(mqres_dir_path):
        match = pattern.match(res_dir)

        if match:
            sample = match.group('sample')
            if sample in reseq_sample_list:           
                txt_path = os.path.join(mqres_dir_path, res_dir, 'test_res.txt')
                bam_path = os.path.join(mqres_dir_path, '../bam_rmdup', f'{sample}.clean.bam')

                sample = sample.split('-')[0]

                if os.path.exists(txt_path) and os.path.exists(bam_path):
                    meta.append({
                        'sample': sample,
                        'deconv_res': txt_path,
                        'clean_bam': bam_path
                    })
                else:
                    print(bam_path)

for batch in ['20260120']:
    mqres_dir_path = f'/appsnew/home/myli/lustre1/bert/DNA_5mC_analysis_pipeline/output/{batch}-XML/bwameth_results/mq_downstream'

    pattern = re.compile(r"^(?P<sample>PTAY.*)_input.bed.gz_testres$")

    for res_dir in os.listdir(mqres_dir_path):
        match = pattern.match(res_dir)

        if match:
            sample = match.group('sample')
            if sample not in reseq_sample_list:          
                txt_path = os.path.join(mqres_dir_path, res_dir, 'test_res.txt')
                bam_path = os.path.join(mqres_dir_path, '../bam_rmdup', f'{sample}.clean.bam')

                sample = sample.split('-')[0]

                if os.path.exists(txt_path) and os.path.exists(bam_path):
                    meta.append({
                        'sample': sample,
                        'deconv_res': txt_path,
                        'clean_bam': bam_path
                    })
                else:
                    print(bam_path)

    pattern = re.compile(r"single_end_(?P<sample>.*)_input.bed.gz_testres$")

    for res_dir in os.listdir(mqres_dir_path):
        match = pattern.match(res_dir)

        if match:
            sample = match.group('sample')
            if sample not in reseq_sample_list:           
                txt_path = os.path.join(mqres_dir_path, res_dir, 'test_res.txt')
                bam_path = os.path.join(mqres_dir_path, '../bam_rmdup', f'{sample}.clean.bam')

                sample = sample.split('-')[0]

                if os.path.exists(txt_path) and os.path.exists(bam_path):
                    meta.append({
                        'sample': sample,
                        'deconv_res': txt_path,
                        'clean_bam': bam_path
                    })
                else:
                    print(bam_path)

meta_df = pd.DataFrame(meta)

meta_df = meta_df[meta_df['sample'] != 'M8']
meta_df = meta_df[~meta_df['sample'].str.contains('UMBS')]

output_samplesheet = '/lustre1/cqyi/syfan/snp_nipt/data/220k_panel/mqres_to_zscore/20260127_mqres_samplesheet.csv'
meta_df.to_csv(output_samplesheet, index=False)